In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


In [2]:
from sentence_transformers import SentenceTransformer
import torch

MODEL_NAME = "BAAI/bge-m3"

model = SentenceTransformer(
    MODEL_NAME,
    device="cuda"
)

model.max_seq_length = 8192

print("Model loaded successfully!")
print("Device:", model.device)
print("GPU:", torch.cuda.get_device_name(0))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Model loaded successfully!
Device: cuda:0
GPU: Tesla T4


In [4]:
!pip install -q faiss-cpu

In [7]:

import os
import re
import sys
import subprocess
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm
from google.colab import files

# 1. Check GPU
assert torch.cuda.is_available(), "GPU is not available. Enable GPU in Colab first."
print("GPU:", torch.cuda.get_device_name(0))

# 2. Upload dataset
uploaded = files.upload()
csv_file = next(
    (name for name in uploaded if name.lower().endswith(".csv")),
    None
)
assert csv_file is not None, "Please upload the CSV dataset."

try:
    df = pd.read_csv(csv_file, encoding="utf-8-sig")
except UnicodeDecodeError:
    df = pd.read_csv(csv_file, encoding="cp1256")

print("Original dataset:", df.shape)

# 3. Validate columns
required = ["case_text", "EVENTS", "REASONING", "RULING"]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Missing columns: {missing}"

# 4. Remove exact duplicate judgments
df_clean = df.drop_duplicates(
    subset=required,
    keep="first"
).copy().reset_index(drop=True)

# 5. Prepare text
def clean_text(text):
    text = str(text)
    text = text.replace("⏎", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n+", "\n", text)
    return text.strip()

df_clean["search_text"] = (
    "الوقائع:\n"
    + df_clean["EVENTS"].fillna("").apply(clean_text)
    + "\n\nأسباب الحكم:\n"
    + df_clean["REASONING"].fillna("").apply(clean_text)
)

# 6. Load tokenizer
from transformers import AutoTokenizer

MODEL_NAME = "BAAI/bge-m3"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 7. Split documents into manageable chunks
MAX_TOKENS = 3000
OVERLAP = 200

chunks = []
chunk_to_doc = []

for doc_id, text in enumerate(tqdm(df_clean["search_text"], desc="Preparing chunks")):
    token_ids = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    if len(token_ids) <= MAX_TOKENS:
        chunks.append(text)
        chunk_to_doc.append(doc_id)
    else:
        start = 0
        while start < len(token_ids):
            end = min(start + MAX_TOKENS, len(token_ids))
            chunk = tokenizer.decode(
                token_ids[start:end],
                skip_special_tokens=True
            )
            chunks.append(chunk)
            chunk_to_doc.append(doc_id)
            start += MAX_TOKENS - OVERLAP

print("Judgments:", len(df_clean))
print("Chunks:", len(chunks))

# 8. Load embedding model
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_NAME, device="cuda")
model.max_seq_length = MAX_TOKENS

# 9. Generate embeddings
embeddings = model.encode(
    chunks,
    batch_size=2,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

# 10. Install FAISS if needed
try:
    import faiss
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip",
        "install", "-q", "faiss-cpu"
    ])
    import faiss

# 11. Build vector index
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

# 12. Save everything
np.save("/content/esnad_embeddings.npy", embeddings)
np.save("/content/esnad_chunk_to_doc.npy", np.array(chunk_to_doc))
faiss.write_index(index, "/content/esnad_faiss.index")
df_clean.to_pickle("/content/esnad_metadata.pkl")

print("\n========== ESNAD RESULTS ==========")
print("Original rows:", len(df))
print("Unique judgments:", len(df_clean))
print("Total chunks:", len(chunks))
print("Embedding shape:", embeddings.shape)
print("FAISS vectors:", index.ntotal)
print("GPU:", torch.cuda.get_device_name(0))
print("\nAll files saved successfully!")


GPU: Tesla T4


Saving ArabiCCR-dataset.csv to ArabiCCR-dataset.csv
Original dataset: (12806, 12)


Preparing chunks:   0%|          | 0/12477 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (10786 > 8192). Running this sequence through the model will result in indexing errors


Judgments: 12477
Chunks: 15532


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/7766 [00:00<?, ?it/s]


========== ESNAD RESULTS ==========
Original rows: 12806
Unique judgments: 12477
Total chunks: 15532
Embedding shape: (15532, 1024)
FAISS vectors: 15532
GPU: Tesla T4

All files saved successfully!


In [8]:
import numpy as np
import pandas as pd

# سؤال قانوني تجريبي
query = "مطالبة شركة بمستحقات عقد توريد بعد تسليم البضائع وامتناع الطرف الآخر عن السداد"

# تحويل السؤال إلى متجه
query_embedding = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

# البحث عن أفضل 200 مقطع
k = min(200, index.ntotal)
scores, indices = index.search(query_embedding, k)

# إزالة تكرار الأحكام والاحتفاظ بأعلى درجة لكل حكم
results = []
seen_docs = set()

for score, chunk_idx in zip(scores[0], indices[0]):
    if chunk_idx < 0:
        continue

    doc_id = int(chunk_to_doc[chunk_idx])

    if doc_id in seen_docs:
        continue

    seen_docs.add(doc_id)
    row = df_clean.iloc[doc_id]

    results.append({
        "similarity": round(float(score), 4),
        "judgment_number": row.get("judgment_number"),
        "case_number": row.get("case_number"),
        "judgment_date": row.get("judgment_date"),
        "city": row.get("city"),
        "details_url": row.get("details_url"),
        "preview": row["search_text"][:500]
    })

    if len(results) == 10:
        break

results_df = pd.DataFrame(results)

print("Search query:", query)
print("Retrieved chunks:", len(indices[0]))
print("Unique judgments returned:", len(results_df))
print("Duplicate judgments in results:", results_df["judgment_number"].duplicated().sum()
      if not results_df.empty else 0)

display(results_df)

Search query: مطالبة شركة بمستحقات عقد توريد بعد تسليم البضائع وامتناع الطرف الآخر عن السداد
Retrieved chunks: 200
Unique judgments returned: 10
Duplicate judgments in results: 0


,similarity,judgment_number,case_number,judgment_date,city,details_url,preview
0,0.7184,5648,5648,1442-06-27,جدة,https://laws.moj.gov.sa/ar/JudicialDecisionsLi...,الوقائع:\nتتلخص وقائع هذه الدعوى بالقدر اللازم...
1,0.6802,1545,1545,1442-11-04,الرياض,https://laws.moj.gov.sa/ar/JudicialDecisionsLi...,الوقائع:\nتتلخص وقائع الدعوى بما يكفي لإصدار ه...
2,0.6773,10876,10876,1442-05-06,الرياض,https://laws.moj.gov.sa/ar/JudicialDecisionsLi...,الوقائع:\nتتلخص وقائع هذه الدعوى في أنه وردت إ...
3,0.6734,6380,6380,1442-08-04,جدة,https://laws.moj.gov.sa/ar/JudicialDecisionsLi...,الوقائع:\nتتلخص وقائع هذه الدعوى في أنه وردت إ...
4,0.6681,2754,2754,1442-08-16,جدة,https://laws.moj.gov.sa/ar/JudicialDecisionsLi...,الوقائع:\nتتلخص وقائع هذه الدعوى بلائحة دعوى أ...
5,0.6627,3274 لعام 1438 هـ,3274 لعام 1438 هـ,1439-03-02,الرياض,https://laws.moj.gov.sa/ar/JudicialDecisionsLi...,الوقائع:\nتتلخص في أن وكيل [Person Name] دعوى ...
6,0.6626,7332,7332,1442-07-16,جدة,https://laws.moj.gov.sa/ar/JudicialDecisionsLi...,الوقائع:\nتقدم المدعي [Person Name]: لي بذمة ا...
7,0.6623,937,937,1442-06-26,جدة,https://laws.moj.gov.sa/ar/JudicialDecisionsLi...,الوقائع:\nتقدم وكيل [Person Name] : تم الاتفاق...
8,0.6622,1812,1812,1442-07-23,جدة,https://laws.moj.gov.sa/ar/JudicialDecisionsLi...,الوقائع:\nتقدم وكيل [Person Name]: قام موكلي ع...
9,0.6614,2557,2557,1440-07-17,الرياض,https://laws.moj.gov.sa/ar/JudicialDecisionsLi...,الوقائع:\nتتلخص وقائع هذه الدعوى بأن وكيل [Per...


In [9]:

import re
import pandas as pd
from collections import Counter

# 1. تجهيز النصوص
text_cols = ["case_text", "EVENTS", "REASONING", "RULING"]

audit_df = df_clean.copy()

audit_df["full_text"] = (
    audit_df[text_cols]
    .fillna("")
    .astype(str)
    .agg("\n".join, axis=1)
)

# 2. مؤشرات درجة التقاضي
degree_patterns = {
    "استئناف": [
        r"محكمة الاستئناف",
        r"الدائرة الاستئنافية",
        r"المحكمة التجارية الاستئنافية",
        r"الحكم المستأنف",
        r"الاستئناف المقدم"
    ],
    "عليا": [
        r"المحكمة العليا",
        r"الدائرة بالمحكمة العليا",
        r"المحكمة التجارية العليا"
    ],
    "ابتدائي - مؤشر صريح": [
        r"المحكمة الابتدائية",
        r"الدائرة الابتدائية",
        r"حكم ابتدائي",
        r"المحكمة التجارية الابتدائية"
    ]
}

def detect_degree(text):
    found = []
    for degree, patterns in degree_patterns.items():
        if any(re.search(p, text) for p in patterns):
            found.append(degree)
    return " | ".join(found) if found else "غير محدد"

audit_df["degree_candidate"] = audit_df["full_text"].apply(detect_degree)

# 3. تصنيف أولي لموضوع النزاع
dispute_patterns = {
    "توريد": [
        r"عقد توريد", r"توريد بضائع", r"توريد مواد",
        r"المورد", r"المورّد"
    ],
    "مقاولات وإنشاءات": [
        r"عقد مقاولة", r"أعمال المقاولة", r"مستخلصات",
        r"مقاول", r"مشروع إنشائي"
    ],
    "مطالبات مالية وديون": [
        r"مطالبة مالية", r"مستحقات مالية",
        r"مبلغ مستحق", r"سداد المبلغ", r"مديونية"
    ],
    "شراكات": [
        r"عقد شراكة", r"الشريك", r"الشركاء",
        r"تصفية الشراكة"
    ],
    "وكالات تجارية": [
        r"وكالة تجارية", r"الوكيل التجاري",
        r"عقد وكالة"
    ],
    "إيجار تجاري": [
        r"عقد إيجار", r"أجرة العقار",
        r"الأجرة المستحقة"
    ],
    "نقل وشحن": [
        r"عقد نقل", r"أجرة النقل",
        r"شحن البضائع", r"النقل التجاري"
    ],
    "أوراق تجارية": [
        r"شيك", r"كمبيالة", r"سند لأمر"
    ],
    "ملكية فكرية": [
        r"علامة تجارية", r"حقوق الملكية الفكرية",
        r"براءة اختراع"
    ],
    "إفلاس": [
        r"إجراء الإفلاس", r"إعادة التنظيم المالي",
        r"التصفية وفق نظام الإفلاس"
    ]
}

def classify_dispute(text):
    found = []
    for category, patterns in dispute_patterns.items():
        if any(re.search(p, text) for p in patterns):
            found.append(category)
    return " | ".join(found) if found else "غير مصنف"

audit_df["dispute_candidate"] = audit_df["full_text"].apply(
    classify_dispute
)

# 4. إحصائيات درجة التقاضي
print("=" * 55)
print("DEGREE OF LITIGATION AUDIT")
print("=" * 55)
print(audit_df["degree_candidate"].value_counts(dropna=False))

# 5. إحصائيات تصنيف النزاع
print("\n" + "=" * 55)
print("DISPUTE CLASSIFICATION AUDIT")
print("=" * 55)

category_counts = Counter()

for value in audit_df["dispute_candidate"]:
    for category in value.split(" | "):
        if category != "غير مصنف":
            category_counts[category] += 1

for category, count in category_counts.most_common():
    print(f"{category}: {count}")

print("غير مصنف:",
      (audit_df["dispute_candidate"] == "غير مصنف").sum())

# 6. عرض عينات للمراجعة
print("\n" + "=" * 55)
print("SAMPLE RESULTS")
print("=" * 55)

sample = audit_df[
    ["judgment_number", "judgment_date",
     "degree_candidate", "dispute_candidate"]
].sample(min(30, len(audit_df)), random_state=42)

display(sample)

# 7. حفظ نتائج الفحص
audit_df.drop(columns=["full_text"]).to_csv(
    "/content/esnad_classification_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nAudit saved successfully:")
print("/content/esnad_classification_audit.csv")
print("Total judgments audited:", len(audit_df))


DEGREE OF LITIGATION AUDIT
degree_candidate
غير محدد                                10704
استئناف                                  1369
عليا                                      169
استئناف | ابتدائي - مؤشر صريح             105
ابتدائي - مؤشر صريح                        64
استئناف | عليا                             54
استئناف | عليا | ابتدائي - مؤشر صريح       11
عليا | ابتدائي - مؤشر صريح                  1
Name: count, dtype: int64

DISPUTE CLASSIFICATION AUDIT
مقاولات وإنشاءات: 4077
أوراق تجارية: 2540
شراكات: 2254
مطالبات مالية وديون: 1985
توريد: 1311
إيجار تجاري: 237
ملكية فكرية: 165
إفلاس: 89
نقل وشحن: 71
وكالات تجارية: 53
غير مصنف: 3848

SAMPLE RESULTS


,judgment_number,judgment_date,degree_candidate,dispute_candidate
429,4530236058,1445-02-01,غير محدد,مقاولات وإنشاءات
9056,1155,1442-11-19,استئناف,مقاولات وإنشاءات
7200,4430238822,1444-04-08,غير محدد,غير مصنف
5716,4430809287,1444-08-05,غير محدد,غير مصنف
5559,4430754413,1444-09-04,استئناف,مطالبات مالية وديون | إيجار تجاري | أوراق تجارية
4190,783/1/ق,1439-05-14,غير محدد,توريد | مقاولات وإنشاءات
4047,2292/1/ق,1439-07-15,غير محدد,شراكات | أوراق تجارية
6832,4430503399,1444-04-24,غير محدد,إيجار تجاري
2167,4089,1441-02-03,غير محدد,مطالبات مالية وديون
7768,4056,1443-03-07,غير محدد,شراكات



Audit saved successfully:
/content/esnad_classification_audit.csv
Total judgments audited: 12477


In [10]:

import re
import pandas as pd

# البحث عن مؤشرات درجة التقاضي في كل حقل على حدة
patterns = {
    "استئناف": r"محكمة الاستئناف|الدائرة الاستئنافية|الحكم المستأنف|الاستئناف المقدم",
    "عليا": r"المحكمة العليا|الدائرة بالمحكمة العليا",
    "ابتدائي": r"المحكمة الابتدائية|الدائرة الابتدائية|حكم ابتدائي"
}

text_fields = ["case_text", "EVENTS", "REASONING", "RULING"]

print("=" * 60)
print("DEGREE EVIDENCE BY TEXT FIELD")
print("=" * 60)

for degree, pattern in patterns.items():
    print(f"\n--- {degree} ---")
    for field in text_fields:
        count = audit_df[field].fillna("").astype(str).str.contains(
            pattern, regex=True
        ).sum()
        print(f"{field}: {count}")

# عرض سياق العبارات التي ظهرت في الأحكام
print("\n" + "=" * 60)
print("REAL TEXT EVIDENCE SAMPLES")
print("=" * 60)

for degree, pattern in patterns.items():
    matched = audit_df[
        audit_df["full_text"].str.contains(pattern, regex=True, na=False)
    ]

    print(f"\n### {degree} | Total: {len(matched)}")

    for _, row in matched.head(5).iterrows():
        text = row["full_text"]
        match = re.search(pattern, text)
        start = max(0, match.start() - 180)
        end = min(len(text), match.end() + 250)

        print("\nJudgment:", row["judgment_number"])
        print("Date:", row["judgment_date"])
        print("Evidence:", text[start:end].replace("\n", " "))

# حفظ العينة
audit_df.to_csv(
    "/content/esnad_degree_evidence.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nAudit completed.")
print("Total judgments:", len(audit_df))


DEGREE EVIDENCE BY TEXT FIELD

--- استئناف ---
case_text: 1538
EVENTS: 1161
REASONING: 659
RULING: 79

--- عليا ---
case_text: 235
EVENTS: 130
REASONING: 122
RULING: 4

--- ابتدائي ---
case_text: 181
EVENTS: 121
REASONING: 70
RULING: 9

REAL TEXT EVIDENCE SAMPLES

### استئناف | Total: 1539

Judgment: 4530004948
Date: 1445-11-16
Evidence: قيته به، ولأن القول المختار في مدة التقادم راجع لاجتهاد القاضي، ولأن مدة سكوت المدعى عن إقامة الدعوى تعد طويلة جدا، لذلك فقد حكم القاضي [Person Name]، فاعترض المدعى، وصدق الحكم من محكمة الاستئناف، ولذلك فإن محكمة (الاستئناف) أوردت في قرارها رقم (275/ق5/ب) وتاريخ 25/6/1429هـ أن دلالة الحال تغني عن دلالة المقال، فالسكوت مدة طويلة عن المطالبة بالملك مع تصرف غيره به دليل عدم صحة الدعوى . وتأسيساً على ما سبق فإني أطلب من فضيلتكم: تثبيت إقراره بأح

Judgment: 4530547483
Date: 1445-06-06
Evidence: ٠٦٥ وتاريخ 5-2-1442 هـ الصادر من دائرتكم الموقّرة والمتضمّن [Person Name] [Person Name] (2893786) مليونين وثمانمائة وثلاثة وتسعون ألف وسبعمائة وستة وثمانون ريال وتم 

In [11]:

import pandas as pd
import re

# اختيار الأحكام التي ظهرت فيها مؤشرات درجة التقاضي
candidates = audit_df[
    audit_df["degree_candidate"] != "غير محدد"
].copy()

# عينة متنوعة
sample_ids = candidates.sample(
    min(20, len(candidates)),
    random_state=21
).index

for i, idx in enumerate(sample_ids, 1):
    row = audit_df.loc[idx]

    print("\n" + "=" * 75)
    print(f"JUDGMENT {i}")
    print("Judgment number:", row["judgment_number"])
    print("Case number:", row["case_number"])
    print("Date:", row["judgment_date"])
    print("Court name:", row["court_name"])
    print("Degree indicators:", row["degree_candidate"])
    print("Dispute indicators:", row["dispute_candidate"])

    text = str(row["case_text"])

    print("\n--- BEGINNING OF CASE TEXT ---")
    print(text[:1200])

    print("\n--- END OF CASE TEXT ---")
    print(text[-800:])

# ملخص القيم الأصلية
print("\n" + "=" * 75)
print("ORIGINAL COURT NAME VALUES")
print(audit_df["court_name"].value_counts(dropna=False).head(20))

print("\nORIGINAL CASE TYPE VALUES")
print(audit_df["case_type"].value_counts(dropna=False).head(20))

print("\nTotal inspected:", len(sample_ids))



JUDGMENT 1
Judgment number: 11070
Case number: 11070
Date: 1443-01-02
Court name: المحكمة التجارية
Degree indicators: استئناف
Dispute indicators: شراكات

--- BEGINNING OF CASE TEXT ---
الحمد لله والصلاة والسلام على رسول ﷲ أما بعد: ⏎ فلدى الدائرة التجارية الخامسة وبناءً على القضية رقم 11070 لعام 1441 هـ ⏎  ⏎ المقامة من/ (...) هوية وطنية (...) ⏎ ضد/ (...) هوية وطنية (...) ⏎  ⏎ (الوقائع) ⏎ تتلخص وقائع هذه الدعوى وبالقدر اللازم لإصدار هذا الحكم في أنه وردت إلى المحكمة التجارية لائحة دعوى مقدمة من المدعي [Person Name]: "أن موضوع الدعوى مطالبة في أتعاب الترافع في قضيتين تم النظر فيهما وإصدار الأحكام من قبل الدائرة الخامسة في المحكمة التجارية في الرياض، وحيث إن نشوء الحق بدأ في تاريخ 04/11/1441هـ حينما فُسخت وكالة المدعي [Person Name] في الأتعاب للقضيتين: القضية الأولى رقم 11624 لعام 1440هـ حيث إن المبلغ المتفق عليه كما هو واضح في الإقرار المرفق والذي كتبه [Person Name] [Person Name] 27/04/2019م الموافق22/08/1440هـ والمبلغ قدره (100000 ريال سعودي) مائة ألف ريال سعودي وحصر طلبه في [Person Nam

In [12]:

import pandas as pd
import re
import os

# ==========================================
# 1. تجهيز البيانات
# ==========================================

data = df_clean.copy()

def normalize(text):
    text = str(text) if pd.notna(text) else ""
    text = text.replace("⏎", "\n")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

for col in ["case_text", "EVENTS", "REASONING", "RULING"]:
    if col not in data.columns:
        data[col] = ""
    data[col] = data[col].apply(normalize)

# ==========================================
# 2. تصنيف درجة التقاضي
# ==========================================

def classify_degree(row):
    text = row["case_text"]
    header = text[:2500]

    explicit_patterns = {
        "عليا": [
            r"الحكم\s+الصادر\s+من\s+المحكمة\s+العليا",
            r"المحكمة\s+العليا",
            r"الدائرة\s+بالمحكمة\s+العليا"
        ],
        "استئناف": [
            r"الحكم\s+الاستئنافي",
            r"الصك\s+الاستئنافي",
            r"محكمة\s+الاستئناف",
            r"الدائرة\s+الاستئنافية",
            r"المحكمة\s+التجارية\s+الاستئنافية"
        ],
        "ابتدائي": [
            r"الحكم\s+الابتدائي",
            r"الصك\s+الابتدائي",
            r"المحكمة\s+الابتدائية",
            r"الدائرة\s+الابتدائية"
        ]
    }

    found = []

    for degree, patterns in explicit_patterns.items():
        if any(re.search(p, header) for p in patterns):
            found.append(degree)

    if len(found) == 1:
        return found[0], "مؤشر صريح في بداية الحكم"

    if len(found) > 1:
        return "متعارض - يحتاج مراجعة", "أكثر من مؤشر في بداية الحكم"

    full_patterns = {
        "عليا": r"المحكمة العليا",
        "استئناف": r"محكمة الاستئناف|الدائرة الاستئنافية|الحكم المستأنف",
        "ابتدائي": r"الحكم الابتدائي|الصك الابتدائي|المحكمة الابتدائية"
    }

    possible = [
        degree for degree, pattern in full_patterns.items()
        if re.search(pattern, text)
    ]

    if possible:
        return "غير محدد", "ذكر درجة أخرى داخل النص - لا يثبت درجة الحكم الحالي"

    return "غير محدد", "لا يوجد مؤشر كافٍ"


degree_results = data.apply(classify_degree, axis=1)
data["degree"] = [x[0] for x in degree_results]
data["degree_evidence"] = [x[1] for x in degree_results]

# ==========================================
# 3. تصنيف نوع النزاع
# ==========================================

dispute_patterns = {
    "توريد": [
        r"عقد التوريد", r"توريد البضائع", r"المواد الموردة",
        r"المورد", r"المشتريات والتوريد"
    ],
    "مقاولات وإنشاءات": [
        r"عقد المقاولة", r"أعمال المقاولات", r"المقاولات",
        r"المشروع الإنشائي", r"تنفيذ المشروع"
    ],
    "شراكات": [
        r"عقد الشراكة", r"الشريك", r"الشركاء",
        r"حصص الشركة", r"تصفية الشركة", r"الشراكة"
    ],
    "مطالبات مالية وديون": [
        r"المطالبة المالية", r"المبلغ المستحق",
        r"المستحقات المالية", r"سداد المبلغ",
        r"الدين المستحق", r"المطالبة بمبلغ"
    ],
    "أوراق تجارية": [
        r"الشيك", r"الكمبيالة", r"السند لأمر",
        r"الأوراق التجارية"
    ],
    "إيجار تجاري": [
        r"عقد الإيجار", r"الأجرة", r"الأجرة المستحقة",
        r"المؤجر", r"المستأجر"
    ],
    "ملكية فكرية": [
        r"الملكية الفكرية", r"العلامة التجارية",
        r"حقوق الملكية", r"حقوق المؤلف", r"براءة الاختراع"
    ],
    "إفلاس": [
        r"الإفلاس", r"إجراء التصفية", r"إعادة التنظيم المالي",
        r"أمين الإفلاس"
    ],
    "نقل وشحن": [
        r"عقد النقل", r"الشحن", r"النقل البحري",
        r"النقل البري", r"بضائع مشحونة"
    ],
    "وكالات تجارية": [
        r"الوكالة التجارية", r"الوكيل التجاري",
        r"عقد الوكالة", r"الوكالات التجارية"
    ]
}

def classify_dispute(row):
    text = row["EVENTS"] + " " + row["REASONING"]
    found = []

    for category, patterns in dispute_patterns.items():
        if any(re.search(p, text) for p in patterns):
            found.append(category)

    return found if found else ["غير مصنف"]


data["dispute_types"] = data.apply(classify_dispute, axis=1)
data["primary_dispute_candidate"] = data["dispute_types"].apply(
    lambda x: x[0] if len(x) == 1 else (
        "متعدد - يحتاج تحديد" if "غير مصنف" not in x else "غير مصنف"
    )
)

# ==========================================
# 4. استخراج نتيجة الحكم
# ==========================================

outcome_patterns = [
    ("عدم جواز نظر الدعوى", r"عدم جواز نظر هذه الدعوى|عدم جواز نظر الدعوى"),
    ("عدم قبول الدعوى", r"عدم قبول الدعوى|عدم قبول هذه الدعوى"),
    ("رفض الدعوى", r"رفض الدعوى|رفض هذه الدعوى"),
    ("إلزام", r"حكمت الدائرة.*?بإلزام|إلزام المدعى عليه|إلزام المدعى عليها"),
    ("ثبوت حق أو شراكة", r"ثبوت شراكة|ثبوت ملكية|ثبوت استحقاق"),
    ("فسخ أو إنهاء", r"فسخ العقد|فسخ الاتفاق|إنهاء العقد"),
    ("حل وتصفية", r"بحل وتصفية|تصفية الشركة"),
    ("تعويض", r"الحكم بالتعويض|إلزام.*?بالتعويض"),
    ("شطب الدعوى", r"شطب الدعوى|قررت الدائرة شطب"),
    ("صلح", r"إثبات الصلح|محضر الصلح|انتهى الطرفان إلى الصلح")
]

def extract_outcome(row):
    text = row["case_text"]
    ruling = row["RULING"]

    # نفضّل الجزء الأخير من الحكم الذي يحتوي منطوقه
    headings = [
        "منطوق الحكم", "نص الحكم", "لذلك", "فلهذه الأسباب",
        "فلهذا", "لهذه الأسباب"
    ]

    positions = [text.rfind(h) for h in headings]
    last_position = max(positions)

    if last_position >= 0:
        relevant = text[last_position:]
    elif len(ruling) >= 30:
        relevant = ruling
    else:
        relevant = text[-1500:]

    found = [
        label for label, pattern in outcome_patterns
        if re.search(pattern, relevant)
    ]

    if found:
        return " | ".join(found), relevant[:700]

    return "غير محدد", relevant[:700]


outcome_results = data.apply(extract_outcome, axis=1)
data["outcome"] = [x[0] for x in outcome_results]
data["outcome_evidence"] = [x[1] for x in outcome_results]

# ==========================================
# 5. استخراج المواد والأنظمة القانونية
# ==========================================

def extract_legal_references(text):
    results = []

    patterns = [
        r"(?:المادة|مادة)\s*(?:رقم\s*)?\(?\s*[\d٠-٩]+\s*\)?",
        r"نظام\s+[ء-ي\s]{2,70}",
        r"اللائحة\s+التنفيذية\s+[ء-ي\s]{2,70}"
    ]

    for pattern in patterns:
        matches = re.findall(pattern, text)
        results.extend(matches)

    cleaned = []
    for item in results:
        item = re.sub(r"\s+", " ", item).strip(" .،؛:")
        if item and item not in cleaned:
            cleaned.append(item)

    return cleaned


data["legal_references"] = data["case_text"].apply(
    extract_legal_references
)

# استخراج سياق كل إشارة قانونية لمراجعتها
def legal_contexts(text):
    contexts = []
    pattern = r"(?:المادة|مادة|نظام|اللائحة التنفيذية)"

    for match in re.finditer(pattern, text):
        start = max(0, match.start() - 100)
        end = min(len(text), match.end() + 160)
        contexts.append(text[start:end])

    return list(dict.fromkeys(contexts))


data["legal_evidence"] = data["case_text"].apply(legal_contexts)

# ==========================================
# 6. إحصائيات شاملة
# ==========================================

print("\n" + "="*65)
print("ESNAD - FINAL DATA AUDIT")
print("="*65)

print("\nDATASET")
print("Total judgments:", len(data))
print("Unique source URLs:", data["details_url"].nunique())

print("\nDEGREE CLASSIFICATION")
print(data["degree"].value_counts(dropna=False).to_string())

print("\nDEGREE EVIDENCE")
print(data["degree_evidence"].value_counts(dropna=False).to_string())

print("\nDISPUTE CLASSIFICATION")
dispute_counts = (
    data["dispute_types"]
    .explode()
    .value_counts()
)
print(dispute_counts.to_string())

print("\nOUTCOME CLASSIFICATION")
print(data["outcome"].value_counts(dropna=False).to_string())

print("\nLEGAL REFERENCES")
print("Judgments with legal references:",
      (data["legal_references"].str.len() > 0).sum())
print("Total extracted references:",
      data["legal_references"].str.len().sum())

# ==========================================
# 7. عينات للمراجعة النهائية
# ==========================================

print("\n" + "="*65)
print("SAMPLE RESULTS")
print("="*65)

sample = data.sample(min(15, len(data)), random_state=42)

for i, (_, row) in enumerate(sample.iterrows(), 1):
    print(f"\n--- SAMPLE {i} ---")
    print("Judgment:", row["judgment_number"])
    print("Degree:", row["degree"])
    print("Degree evidence:", row["degree_evidence"])
    print("Dispute:", row["dispute_types"])
    print("Outcome:", row["outcome"])
    print("Legal references:", row["legal_references"][:10])
    print("Outcome evidence:", row["outcome_evidence"][:350])

# ==========================================
# 8. حفظ المخرجات
# ==========================================

export = data.copy()

for col in ["dispute_types", "legal_references", "legal_evidence"]:
    export[col] = export[col].apply(
        lambda x: " || ".join(x) if isinstance(x, list) else x
    )

csv_path = "/content/esnad_arabiccr_enriched.csv"
export.to_csv(csv_path, index=False, encoding="utf-8-sig")

summary_path = "/content/esnad_final_audit.txt"

with open(summary_path, "w", encoding="utf-8") as f:
    f.write("ESNAD - FINAL AUDIT\n")
    f.write("="*50 + "\n\n")
    f.write(f"Total judgments: {len(data)}\n")
    f.write(f"Unique URLs: {data['details_url'].nunique()}\n\n")

    for title, series in [
        ("DEGREE", data["degree"].value_counts()),
        ("DEGREE EVIDENCE", data["degree_evidence"].value_counts()),
        ("DISPUTES", dispute_counts),
        ("OUTCOMES", data["outcome"].value_counts())
    ]:
        f.write(f"\n{title}\n")
        f.write(series.to_string())
        f.write("\n")

    f.write("\n\nLEGAL REFERENCES\n")
    f.write(
        f"Judgments with references: "
        f"{(data['legal_references'].str.len() > 0).sum()}\n"
    )
    f.write(
        f"Total references: {data['legal_references'].str.len().sum()}\n"
    )

print("\n" + "="*65)
print("FILES SAVED")
print(csv_path)
print(summary_path)
print("="*65)



ESNAD - FINAL DATA AUDIT

DATASET
Total judgments: 12477
Unique source URLs: 12477

DEGREE CLASSIFICATION
degree
غير محدد                 11631
استئناف                    660
متعارض - يحتاج مراجعة       66
ابتدائي                     60
عليا                        60

DEGREE EVIDENCE
degree_evidence
لا يوجد مؤشر كافٍ                                      10695
ذكر درجة أخرى داخل النص - لا يثبت درجة الحكم الحالي      936
مؤشر صريح في بداية الحكم                                 780
أكثر من مؤشر في بداية الحكم                               66

DISPUTE CLASSIFICATION
dispute_types
غير مصنف               4969
شراكات                 3023
مطالبات مالية وديون    2014
أوراق تجارية           1824
مقاولات وإنشاءات       1143
إيجار تجاري             912
توريد                   832
ملكية فكرية             301
إفلاس                   292
نقل وشحن                261
وكالات تجارية           120

OUTCOME CLASSIFICATION
outcome
إلزام                                                         4438
غير محدد 

In [13]:

import re
import json
import pandas as pd
import numpy as np
from collections import Counter

# ============================================================
# ESNAD - IMPROVED OUTCOME + LEGAL REFERENCES EXTRACTION
# Requires: df_clean
# Does NOT rebuild embeddings or FAISS
# ============================================================

if "df_clean" not in globals():
    raise NameError("لم أجد df_clean. شغّلي الخلية في جلسة Colab التي تحتوي على بيانات ArabiCCR.")

data = df_clean.copy()

required = ["case_text", "EVENTS", "REASONING", "RULING"]
missing = [c for c in required if c not in data.columns]
if missing:
    raise ValueError(f"أعمدة ناقصة في البيانات: {missing}")

# ----------------------------
# 1. Arabic text normalization
# ----------------------------

def normalize_ar(text):
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""

    text = str(text)
    text = re.sub(r"[\u064B-\u065F\u0670\u0640]", "", text)
    text = re.sub(r"[أإآٱ]", "ا", text)
    text = text.replace("ى", "ي")
    text = text.replace("ؤ", "و").replace("ئ", "ي")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def western_digits(text):
    table = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
    return text.translate(table)


# ---------------------------------
# 2. Extract the operative ruling
# ---------------------------------

RULING_HEADINGS = [
    r"منطوق الحكم",
    r"نص الحكم",
    r"لذلك\s*حكمت",
    r"لهذه الأسباب",
    r"لهذه الاسباب",
    r"حكمت الدائرة",
    r"حكمت المحكمة",
    r"قررت الدائرة",
    r"قضت الدائرة",
]

HEADING_RE = re.compile(
    "|".join(RULING_HEADINGS),
    flags=re.IGNORECASE
)

def get_ruling_candidate(row):
    ruling_col = normalize_ar(row.get("RULING", ""))
    case_text = normalize_ar(row.get("case_text", ""))

    candidates = []

    # RULING is a dedicated source column, but may contain
    # incomplete or noisy text. Keep it as one candidate.
    if ruling_col:
        candidates.append(("RULING", ruling_col[-5000:]))

    # Find the last explicit ruling heading in the full judgment.
    matches = list(HEADING_RE.finditer(case_text))
    if matches:
        last_match = matches[-1]
        segment = case_text[last_match.start():last_match.start() + 3500]
        candidates.append(("case_text_last_heading", segment))

    # If no heading exists, inspect only the end of the judgment.
    if not candidates and case_text:
        candidates.append(("case_text_end", case_text[-2500:]))

    # Prefer the candidate with the clearest operative language.
    operative_terms = [
        "حكمت", "قضت", "بالزام", "الزام", "رفض الدعوي",
        "رفض الدعوى", "عدم قبول", "عدم جواز", "عدم اختصاص",
        "اثبات الصلح", "فسخ", "شطب"
    ]

    ranked = []
    for source, candidate in candidates:
        score = sum(term in candidate for term in operative_terms)
        ranked.append((score, source, candidate))

    if not ranked:
        return "", "لا يوجد نص للحكم"

    ranked.sort(key=lambda x: x[0], reverse=True)
    score, source, candidate = ranked[0]

    if score == 0:
        return candidate, "نص مرشح دون مؤشرات حكم واضحة"

    return candidate, source


# ---------------------------------
# 3. Multi-label outcome extraction
# ---------------------------------

OUTCOME_PATTERNS = {
    "إلزام": [
        r"بالزام",
        r"الزام\s+(?:المدعى|المدعي عليه|المدعي عليها)",
        r"حكمت.{0,40}بالزام",
        r"قضت.{0,40}بالزام",
        r"الزام.{0,100}(?:بان|بأن)\s+(?:يدفع|تدفع|يسدد|تسدد)",
        r"الزام.{0,100}(?:بتمكين|بتسليم|بإخلاء|باخلاء)",
    ],
    "رفض الدعوى": [
        r"رفض الدعوي",
        r"رفض الدعوى",
        r"برفض الدعوي",
        r"برفض الدعوى",
        r"رد الدعوي",
        r"رد الدعوى",
        r"رفض طلب",
        r"برفض طلب",
    ],
    "عدم قبول الدعوى": [
        r"عدم قبول الدعوي",
        r"عدم قبول الدعوى",
        r"بعدم قبول الدعوي",
        r"بعدم قبول الدعوى",
        r"عدم قبول الطلب",
        r"بعدم قبول الطلب",
    ],
    "عدم جواز نظر الدعوى": [
        r"عدم جواز نظر الدعوي",
        r"عدم جواز نظر الدعوى",
        r"بعدم جواز نظر الدعوي",
        r"بعدم جواز نظر الدعوى",
    ],
    "عدم الاختصاص": [
        r"عدم الاختصاص",
        r"بعدم اختصاص",
        r"عدم اختصاص الدائرة",
        r"عدم اختصاص المحكمة",
    ],
    "فسخ أو إنهاء": [
        r"فسخ العقد",
        r"فسخ الاتفاق",
        r"انهاء العقد",
        r"إنهاء العقد",
        r"بفسخ",
        r"بانهاء",
    ],
    "صلح": [
        r"اثبات الصلح",
        r"إثبات الصلح",
        r"اثبات اتفاق الصلح",
        r"الصلح المبرم",
        r"تصالح الطرفان",
        r"اتفاق الطرفين على الصلح",
    ],
    "شطب الدعوى": [
        r"شطب الدعوي",
        r"شطب الدعوى",
        r"بشطب الدعوي",
        r"بشطب الدعوى",
    ],
    "ثبوت حق أو شراكة": [
        r"ثبوت الشراكه",
        r"ثبوت الشراكة",
        r"ثبوت حق المدعي",
        r"ثبوت استحقاق",
        r"اثبات الشراكه",
        r"اثبات الشراكة",
    ],
    "حل وتصفية": [
        r"حل الشركة",
        r"حل الشركه",
        r"تصفية الشركة",
        r"تصفية الشركه",
        r"حل وتصفية",
    ],
}

def extract_outcomes(text):
    text = normalize_ar(text)
    found = []

    for label, patterns in OUTCOME_PATTERNS.items():
        for pattern in patterns:
            if re.search(pattern, text):
                found.append(label)
                break

    # Avoid counting acceptance as an outcome unless explicitly stated.
    # "عدم قبول الدعوى" is already handled separately.
    return found


def outcome_evidence(text, outcomes, window=220):
    text = normalize_ar(text)
    evidence = {}

    for label in outcomes:
        patterns = OUTCOME_PATTERNS[label]
        snippets = []

        for pattern in patterns:
            for match in re.finditer(pattern, text):
                start = max(0, match.start() - 90)
                end = min(len(text), match.end() + 130)
                snippets.append(text[start:end])

        # Keep up to two short examples for each outcome.
        evidence[label] = snippets[:2]

    return evidence


ruling_results = data.apply(get_ruling_candidate, axis=1)
data["ruling_candidate"] = [x[0] for x in ruling_results]
data["ruling_source"] = [x[1] for x in ruling_results]

data["outcomes_list"] = data["ruling_candidate"].apply(extract_outcomes)
data["outcome_improved"] = data["outcomes_list"].apply(
    lambda x: " | ".join(x) if x else "غير محدد"
)
data["outcome_evidence_improved"] = data.apply(
    lambda row: json.dumps(
        outcome_evidence(row["ruling_candidate"], row["outcomes_list"]),
        ensure_ascii=False
    ),
    axis=1
)

# ---------------------------------
# 4. Legal system name extraction
# ---------------------------------

# Controlled list: only recognize system names explicitly listed here.
# Expand this list later after reviewing the actual dataset.
LEGAL_SYSTEMS = [
    "نظام المحاكم التجارية",
    "اللائحة التنفيذية لنظام المحاكم التجارية",
    "نظام الشركات",
    "نظام الإثبات",
    "نظام الاثبات",
    "نظام التحكيم",
    "نظام الإفلاس",
    "نظام الافلاس",
    "نظام المعاملات المدنية",
    "نظام المرافعات الشرعية",
    "نظام التنفيذ",
    "نظام الأوراق التجارية",
    "نظام الاوراق التجارية",
    "نظام المنافسة",
    "نظام الامتياز التجاري",
    "نظام الوكالات التجارية",
    "نظام العلامات التجارية",
    "نظام حماية حقوق المؤلف",
    "نظام براءات الاختراع",
    "نظام التجارة الإلكترونية",
    "نظام التعاملات الإلكترونية",
    "نظام السجل التجاري",
    "نظام الأسماء التجارية",
    "نظام الاستثمار",
    "نظام العمل",
    "نظام الرهن التجاري",
    "نظام الملكية الفكرية",
    "نظام مكافحة الغش التجاري",
    "نظام مكافحة التستر",
    "نظام ضريبة القيمة المضافة",
]

# Normalize list but retain display form.
LEGAL_SYSTEMS_NORMALIZED = {
    normalize_ar(name): name for name in LEGAL_SYSTEMS
}

def extract_legal_systems(text):
    normalized = normalize_ar(text)
    found = []

    # Longer names first to avoid partial matching.
    names = sorted(
        LEGAL_SYSTEMS_NORMALIZED.keys(),
        key=len,
        reverse=True
    )

    for normalized_name in names:
        if normalized_name in normalized:
            display_name = LEGAL_SYSTEMS_NORMALIZED[normalized_name]
            if display_name not in found:
                found.append(display_name)

    return found


# ---------------------------------
# 5. Legal article number extraction
# ---------------------------------

ARTICLE_PATTERN = re.compile(
    r"(?:المادة|مادة)\s*(?:رقم\s*)?"
    r"[\(\[]?\s*([0-9٠-٩]{1,3})\s*[\)\]]?"
    r"(?:\s*(?:مكرر|المكررة))?",
    flags=re.IGNORECASE
)

def extract_articles(text):
    normalized = normalize_ar(text)
    normalized = western_digits(normalized)

    found = []
    for match in ARTICLE_PATTERN.finditer(normalized):
        number = match.group(1)
        label = f"المادة {number}"
        if label not in found:
            found.append(label)

    return found


# ---------------------------------
# 6. Evidence snippets for references
# ---------------------------------

def get_reference_evidence(text, terms, max_per_term=2):
    text = normalize_ar(text)
    output = {}

    for term in terms:
        snippets = []
        search_term = normalize_ar(term)

        start_pos = 0
        while len(snippets) < max_per_term:
            pos = text.find(search_term, start_pos)
            if pos == -1:
                break

            start = max(0, pos - 100)
            end = min(len(text), pos + len(search_term) + 140)
            snippets.append(text[start:end])
            start_pos = pos + len(search_term)

        output[term] = snippets

    return output


# Use reasoning + full judgment for legal references.
def legal_source_text(row):
    parts = [
        normalize_ar(row.get("REASONING", "")),
        normalize_ar(row.get("case_text", "")),
    ]
    return "\n".join(part for part in parts if part)


data["legal_source_text"] = data.apply(legal_source_text, axis=1)

data["legal_systems"] = data["legal_source_text"].apply(
    extract_legal_systems
)
data["legal_articles"] = data["legal_source_text"].apply(
    extract_articles
)

data["legal_systems_evidence"] = data.apply(
    lambda row: json.dumps(
        get_reference_evidence(
            row["legal_source_text"],
            row["legal_systems"]
        ),
        ensure_ascii=False
    ),
    axis=1
)

data["legal_articles_evidence"] = data.apply(
    lambda row: json.dumps(
        get_reference_evidence(
            row["legal_source_text"],
            row["legal_articles"]
        ),
        ensure_ascii=False
    ),
    axis=1
)

# ---------------------------------
# 7. Summary report
# ---------------------------------

print("=" * 65)
print("ESNAD - IMPROVED OUTCOME & LEGAL REFERENCES AUDIT")
print("=" * 65)

print("\nOUTCOME DISTRIBUTION")
print(data["outcome_improved"].value_counts().to_string())

print("\nOUTCOME COVERAGE")
outcome_found = (data["outcome_improved"] != "غير محدد").sum()
print(f"Judgments with extracted outcome: {outcome_found:,}")
print(f"Judgments without clear outcome: {len(data)-outcome_found:,}")
print(f"Coverage: {outcome_found/len(data)*100:.2f}%")

multi_outcome = data["outcomes_list"].apply(len).gt(1).sum()
print(f"Judgments with multiple outcomes: {multi_outcome:,}")

print("\nLEGAL SYSTEMS")
system_counts = Counter(
    system
    for systems in data["legal_systems"]
    for system in systems
)
for name, count in system_counts.most_common():
    print(f"{name}: {count:,}")

print("\nLEGAL ARTICLES")
article_counts = Counter(
    article
    for articles in data["legal_articles"]
    for article in articles
)
print(f"Unique article labels: {len(article_counts):,}")
print(f"Total article mentions: {sum(article_counts.values()):,}")
for name, count in article_counts.most_common(25):
    print(f"{name}: {count:,}")

systems_covered = data["legal_systems"].apply(bool).sum()
articles_covered = data["legal_articles"].apply(bool).sum()

print("\nLEGAL REFERENCE COVERAGE")
print(f"Judgments with recognized legal systems: {systems_covered:,}")
print(f"Judgments with article numbers: {articles_covered:,}")

# ---------------------------------
# 8. Print review samples
# ---------------------------------

print("\n" + "=" * 65)
print("SAMPLES: MULTIPLE OUTCOMES")
print("=" * 65)

multi_samples = data[data["outcomes_list"].apply(len).gt(1)].head(8)

for _, row in multi_samples.iterrows():
    print("\nJudgment:", row.get("judgment_number"))
    print("Outcomes:", row["outcomes_list"])
    print("Source:", row["ruling_source"])
    print("Evidence:", row["outcome_evidence_improved"][:1000])

print("\n" + "=" * 65)
print("SAMPLES: LEGAL REFERENCES")
print("=" * 65)

legal_samples = data[
    data["legal_systems"].apply(bool)
    | data["legal_articles"].apply(bool)
].head(10)

for _, row in legal_samples.iterrows():
    print("\nJudgment:", row.get("judgment_number"))
    print("Systems:", row["legal_systems"])
    print("Articles:", row["legal_articles"])
    print("System evidence:", row["legal_systems_evidence"][:700])
    print("Article evidence:", row["legal_articles_evidence"][:700])

# ---------------------------------
# 9. Save improved dataset
# ---------------------------------

# Convert list/dict columns to JSON strings for reliable CSV storage.
export = data.copy()

for col in [
    "outcomes_list",
    "legal_systems",
    "legal_articles",
]:
    export[col] = export[col].apply(
        lambda x: json.dumps(x, ensure_ascii=False)
    )

output_csv = "/content/esnad_arabiccr_improved.csv"
output_report = "/content/esnad_improved_extraction_report.txt"

export.to_csv(output_csv, index=False, encoding="utf-8-sig")

with open(output_report, "w", encoding="utf-8") as f:
    f.write("ESNAD IMPROVED EXTRACTION AUDIT\n")
    f.write("=" * 60 + "\n\n")
    f.write("OUTCOME DISTRIBUTION\n")
    f.write(data["outcome_improved"].value_counts().to_string())
    f.write("\n\nLEGAL SYSTEMS\n")
    f.write(pd.Series(dict(system_counts)).to_string())
    f.write("\n\nLEGAL ARTICLES - TOP 50\n")
    f.write(pd.Series(dict(article_counts)).sort_values(
        ascending=False
    ).head(50).to_string())
    f.write("\n\nMULTI-OUTCOME SAMPLES\n")
    for _, row in multi_samples.iterrows():
        f.write(f"\nJudgment: {row.get('judgment_number')}\n")
        f.write(f"Outcomes: {row['outcomes_list']}\n")
        f.write(f"Evidence: {row['outcome_evidence_improved']}\n")

print("\n" + "=" * 65)
print("FILES SAVED")
print(output_csv)
print(output_report)
print("=" * 65)


ESNAD - IMPROVED OUTCOME & LEGAL REFERENCES AUDIT

OUTCOME DISTRIBUTION
outcome_improved
إلزام                                                         5398
غير محدد                                                      3930
رفض الدعوى                                                    1634
عدم الاختصاص                                                   598
عدم قبول الدعوى                                                432
عدم جواز نظر الدعوى                                            118
إلزام | فسخ أو إنهاء                                            99
إلزام | رفض الدعوى                                              75
فسخ أو إنهاء                                                    67
إلزام | عدم قبول الدعوى                                         30
حل وتصفية                                                       18
إلزام | عدم الاختصاص                                             9
رفض الدعوى | فسخ أو إنهاء                                        9
ثبوت حق أو شراكة                        

In [14]:
# ============================================================
# ESNAD - REFINED OUTCOME & LEGAL REFERENCE AUDIT
# يعتمد على df_clean الموجود مسبقًا
# لا يعيد بناء Embeddings أو FAISS
# ============================================================

import pandas as pd
import re
import json
from collections import Counter

assert "df_clean" in globals(), "لم أجد df_clean. شغّلي خلايا تجهيز البيانات أولًا."

df_audit = df_clean.copy().reset_index(drop=True)

# ------------------------------------------------------------
# 1) Arabic text normalization
# ------------------------------------------------------------

def norm_ar(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ى", "ي").replace("ة", "ه")
    text = text.replace("ؤ", "و").replace("ئ", "ي")
    text = re.sub(r"[ًٌٍَُِّْـ]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def to_western_digits(text):
    trans = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
    return text.translate(trans)

# ------------------------------------------------------------
# 2) Refined outcome extraction
#    Only inspect text following explicit judicial ruling anchors
# ------------------------------------------------------------

OUTCOME_PATTERNS = {
    "إلزام": [
        r"بالزام", r"الزام المدعي", r"الزام المدعي عليه",
        r"الزام المدعي عليها", r"بان يدفع", r"بان تدفع",
        r"بان يؤدي", r"بان تؤدي", r"الزام الطرف"
    ],
    "رفض الدعوى أو الطلب": [
        r"رفض الدعوي", r"رفض طلب", r"رفض الدعوى"
    ],
    "عدم قبول الدعوى": [
        r"عدم قبول الدعوي", r"بعدم قبول الدعوي"
    ],
    "عدم جواز نظر الدعوى": [
        r"عدم جواز نظر الدعوي", r"بعدم جواز نظر الدعوي"
    ],
    "عدم الاختصاص": [
        r"عدم الاختصاص", r"عدم اختصاص المحكمه",
        r"عدم اختصاص الدائره", r"عدم اختصاص الدائرة"
    ],
    "فسخ أو إنهاء": [
        r"فسخ العقد", r"فسخ الاتفاق", r"انهاء العقد",
        r"انهاء الاتفاق"
    ],
    "صلح": [
        r"اثبات الصلح", r"اعتماد الصلح", r"الصلح بين الطرفين"
    ],
    "شطب الدعوى": [
        r"شطب الدعوي", r"شطب القضية"
    ],
    "حل وتصفية": [
        r"حل الشركه", r"تصفية الشركه", r"حل وتصفية"
    ],
    "ثبوت حق أو شراكة": [
        r"ثبوت الشراكه", r"ثبوت حق", r"اثبات الشراكه"
    ]
}

RULING_ANCHORS = [
    "حكمت الدايره",
    "حكمت المحكمه",
    "قضت الدايره",
    "قضت المحكمه",
    "لذلك حكمت",
    "فلهذه الاسباب حكمت",
    "عليه حكمت المحكمه",
    "عليه حكمت الدايره"
]

def get_ruling_windows(text):
    text = norm_ar(text)
    windows = []

    for anchor in RULING_ANCHORS:
        start = 0
        while True:
            pos = text.find(anchor, start)
            if pos == -1:
                break

            # استبعاد بعض الإشارات التي تبدو سردًا لحكم سابق
            before = text[max(0, pos-180):pos]
            historical_markers = [
                "سبق ان صدر", "صدر حكم", "حصل علي حكم",
                "الحكم السابق", "الحكم الصادر", "بموجب حكم"
            ]

            if not any(x in before for x in historical_markers):
                windows.append(text[pos:pos+1800])

            start = pos + len(anchor)

    # إزالة النوافذ المتكررة والمتداخلة
    unique = []
    for w in windows:
        if not any(w[:250] == old[:250] for old in unique):
            unique.append(w)

    return unique

def classify_refined_outcome(ruling_text):
    windows = get_ruling_windows(ruling_text)
    found = {}
    for label, patterns in OUTCOME_PATTERNS.items():
        for window in windows:
            matched = [p for p in patterns if re.search(p, window)]
            if matched:
                found[label] = window[:650]
                break

    labels = list(found.keys())

    if not labels:
        status = "غير محدد"
    elif len(labels) == 1:
        status = labels[0]
    else:
        status = " | ".join(labels)

    return status, labels, found, len(windows)

outcome_results = df_audit["RULING"].apply(classify_refined_outcome)

df_audit["outcome_refined"] = outcome_results.apply(lambda x: x[0])
df_audit["outcomes_refined_list"] = outcome_results.apply(
    lambda x: json.dumps(x[1], ensure_ascii=False)
)
df_audit["outcome_refined_evidence"] = outcome_results.apply(
    lambda x: json.dumps(x[2], ensure_ascii=False)
)
df_audit["ruling_anchor_count"] = outcome_results.apply(lambda x: x[3])

# ------------------------------------------------------------
# 3) Legal systems: controlled names and aliases
# ------------------------------------------------------------

LEGAL_SYSTEMS = {
    "نظام المحاكم التجارية": [
        "نظام المحاكم التجارية"
    ],
    "اللائحة التنفيذية لنظام المحاكم التجارية": [
        "اللائحة التنفيذية لنظام المحاكم التجارية",
        "لايحة نظام المحاكم التجارية",
        "اللائحة التنفيذية للمحاكم التجارية"
    ],
    "نظام المرافعات الشرعية": [
        "نظام المرافعات الشرعية"
    ],
    "نظام الشركات": [
        "نظام الشركات"
    ],
    "نظام الإثبات": [
        "نظام الاثبات"
    ],
    "نظام الإفلاس": [
        "نظام الافلاس"
    ],
    "نظام التحكيم": [
        "نظام التحكيم"
    ],
    "نظام التنفيذ": [
        "نظام التنفيذ"
    ],
    "نظام المعاملات المدنية": [
        "نظام المعاملات المدنية"
    ],
    "نظام الأوراق التجارية": [
        "نظام الاوراق التجارية"
    ],
    "نظام العلامات التجارية": [
        "نظام العلامات التجارية"
    ],
    "نظام الأسماء التجارية": [
        "نظام الاسماء التجارية"
    ],
    "نظام حماية حقوق المؤلف": [
        "نظام حماية حقوق المؤلف"
    ],
    "نظام براءات الاختراع": [
        "نظام براءات الاختراع"
    ],
    "نظام الوكالات التجارية": [
        "نظام الوكالات التجارية"
    ],
    "نظام الامتياز التجاري": [
        "نظام الامتياز التجاري"
    ],
    "نظام العمل": [
        "نظام العمل"
    ],
    "نظام السجل التجاري": [
        "نظام السجل التجاري"
    ],
    "نظام الاستثمار": [
        "نظام الاستثمار"
    ],
    "نظام التعاملات الإلكترونية": [
        "نظام التعاملات الالكترونية"
    ],
    "نظام التجارة الإلكترونية": [
        "نظام التجارة الالكترونية"
    ],
    "نظام مكافحة التستر": [
        "نظام مكافحة التستر"
    ],
    "نظام مكافحة الغش التجاري": [
        "نظام مكافحة الغش التجاري"
    ],
    "نظام المنافسة": [
        "نظام المنافسة"
    ],
    "نظام ضريبة القيمة المضافة": [
        "نظام ضريبة القيمة المضافة"
    ],
    "نظام الملكية الفكرية": [
        "نظام الملكية الفكرية"
    ]
}

SYSTEM_PATTERNS = {}

for system, aliases in LEGAL_SYSTEMS.items():
    SYSTEM_PATTERNS[system] = [
        norm_ar(alias) for alias in aliases
    ]

# ------------------------------------------------------------
# 4) Article extraction and contextual system association
# ------------------------------------------------------------

ARTICLE_REGEX = re.compile(
    r"الماده\s*[:：]?\s*"
    r"(?:رقم\s*)?"
    r"[\(（]?\s*([0-9٠-٩]{1,4})\s*[\)）]?"
    r"(?:\s*/\s*([0-9٠-٩]{1,3}))?"
)

def find_system_mentions(text):
    text = norm_ar(text)
    mentions = []

    for system, aliases in SYSTEM_PATTERNS.items():
        for alias in aliases:
            start = 0
            while True:
                pos = text.find(alias, start)
                if pos == -1:
                    break

                mentions.append({
                    "system": system,
                    "start": pos,
                    "end": pos + len(alias)
                })
                start = pos + len(alias)

    return mentions

def extract_references_from_text(text, source_name):
    text = norm_ar(text)
    if not text:
        return [], []

    systems = find_system_mentions(text)
    articles = []

    for match in ARTICLE_REGEX.finditer(text):
        number = to_western_digits(match.group(1))
        sub_number = match.group(2)

        article_label = f"المادة {number}"
        if sub_number:
            article_label += f"/{to_western_digits(sub_number)}"

        start = match.start()
        end = match.end()

        # سياق المادة قبلها وبعدها
        left = max(0, start - 300)
        right = min(len(text), end + 300)
        context = text[left:right]

        # اختيار أقرب نظام مذكور ضمن السياق
        nearby_systems = []
        for system_mention in systems:
            distance = min(
                abs(system_mention["start"] - start),
                abs(system_mention["end"] - end)
            )
            if distance <= 300:
                nearby_systems.append((distance, system_mention["system"]))

        nearby_systems.sort(key=lambda x: x[0])

        linked_system = nearby_systems[0][1] if nearby_systems else None

        articles.append({
            "article": article_label,
            "system": linked_system,
            "source": source_name,
            "context": context
        })

    system_records = []
    for mention in systems:
        left = max(0, mention["start"] - 220)
        right = min(len(text), mention["end"] + 220)

        system_records.append({
            "system": mention["system"],
            "source": source_name,
            "context": text[left:right]
        })

    return system_records, articles

# ------------------------------------------------------------
# 5) Extract references separately by text field
# ------------------------------------------------------------

all_system_records = []
all_article_records = []
per_judgment_systems = []
per_judgment_articles = []

for idx, row in df_audit.iterrows():
    judgment_id = str(row.get("judgment_number", idx))

    judgment_systems = []
    judgment_articles = []

    # أسباب الحكم هي المصدر الأهم للاستدلال النظامي.
    # الوقائع والمنطوق تحفظ منفصلة ولا تدمج معها.
    for col, source_name in [
        ("REASONING", "أسباب الحكم"),
        ("RULING", "المنطوق/حقل RULING"),
        ("EVENTS", "وقائع القضية")
    ]:
        if col not in df_audit.columns:
            continue

        sys_records, art_records = extract_references_from_text(
            row[col], source_name
        )

        for item in sys_records:
            item["judgment_number"] = judgment_id
            all_system_records.append(item)
            judgment_systems.append(item)

        for item in art_records:
            item["judgment_number"] = judgment_id
            all_article_records.append(item)
            judgment_articles.append(item)

    per_judgment_systems.append(sorted(set(
        x["system"] for x in judgment_systems
    )))

    per_judgment_articles.append(judgment_articles)

df_audit["legal_systems_refined"] = [
    json.dumps(x, ensure_ascii=False) for x in per_judgment_systems
]

df_audit["legal_articles_refined"] = [
    json.dumps(x, ensure_ascii=False) for x in per_judgment_articles
]

# ------------------------------------------------------------
# 6) Build reference tables
# ------------------------------------------------------------

systems_df = pd.DataFrame(all_system_records)
articles_df = pd.DataFrame(all_article_records)

if not systems_df.empty:
    systems_summary = (
        systems_df.groupby("system")
        .agg(
            mentions=("system", "size"),
            judgments=("judgment_number", "nunique")
        )
        .sort_values("judgments", ascending=False)
    )
else:
    systems_summary = pd.DataFrame()

if not articles_df.empty:
    articles_summary = (
        articles_df.groupby(["article", "system"], dropna=False)
        .agg(
            mentions=("article", "size"),
            judgments=("judgment_number", "nunique")
        )
        .reset_index()
        .sort_values(["judgments", "mentions"], ascending=False)
    )
else:
    articles_summary = pd.DataFrame()

# ------------------------------------------------------------
# 7) Print audit report
# ------------------------------------------------------------

print("=" * 65)
print("ESNAD - REFINED OUTCOME & LEGAL REFERENCES AUDIT")
print("=" * 65)

print("\nOUTCOME DISTRIBUTION")
print(df_audit["outcome_refined"].value_counts().to_string())

identified = (df_audit["outcome_refined"] != "غير محدد").sum()
multiple = df_audit["outcomes_refined_list"].apply(
    lambda x: len(json.loads(x)) > 1
).sum()

print("\nOUTCOME COVERAGE")
print("Total judgments:", len(df_audit))
print("Judgments with anchored outcome:", identified)
print("Judgments without anchored outcome:", len(df_audit) - identified)
print("Coverage:", round(identified / len(df_audit) * 100, 2), "%")
print("Judgments with multiple outcomes:", multiple)
print("Judgments with no explicit ruling anchor:",
      (df_audit["ruling_anchor_count"] == 0).sum())

print("\nLEGAL SYSTEMS")
if not systems_summary.empty:
    print(systems_summary.to_string())
else:
    print("No systems extracted.")

print("\nARTICLE REFERENCES BY SYSTEM")
if not articles_summary.empty:
    print(articles_summary.head(35).to_string(index=False))
else:
    print("No article references extracted.")

print("\nREFERENCE LINKING")
if not articles_df.empty:
    linked = articles_df["system"].notna().sum()
    unlinked = articles_df["system"].isna().sum()
    print("Total article mentions:", len(articles_df))
    print("Articles linked to a nearby system:", linked)
    print("Articles without a nearby system:", unlinked)
    print("Linking rate:", round(linked / len(articles_df) * 100, 2), "%")

print("\nSAMPLES: ARTICLE REFERENCES WITH CONTEXT")
if not articles_df.empty:
    sample_articles = articles_df.sample(
        min(12, len(articles_df)), random_state=42
    )
    for _, item in sample_articles.iterrows():
        print("\nJudgment:", item["judgment_number"])
        print("Article:", item["article"])
        print("System:", item["system"])
        print("Source:", item["source"])
        print("Context:", item["context"][:450])

print("\nSAMPLES: REFINED OUTCOMES")
sample_outcomes = df_audit[
    df_audit["outcome_refined"] != "غير محدد"
].sample(
    min(12, identified), random_state=42
)

for _, row in sample_outcomes.iterrows():
    print("\nJudgment:", row.get("judgment_number"))
    print("Outcome:", row["outcome_refined"])
    print("Anchor count:", row["ruling_anchor_count"])
    print("Evidence:", row["outcome_refined_evidence"][:900])

# ------------------------------------------------------------
# 8) Save outputs
# ------------------------------------------------------------

df_audit.to_csv(
    "/content/esnad_arabiccr_refined_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

if not systems_df.empty:
    systems_df.to_csv(
        "/content/esnad_legal_system_mentions.csv",
        index=False,
        encoding="utf-8-sig"
    )

if not articles_df.empty:
    articles_df.to_csv(
        "/content/esnad_legal_article_references.csv",
        index=False,
        encoding="utf-8-sig"
    )

print("\n" + "=" * 65)
print("FILES SAVED")
print("/content/esnad_arabiccr_refined_audit.csv")
print("/content/esnad_legal_system_mentions.csv")
print("/content/esnad_legal_article_references.csv")
print("=" * 65)

ESNAD - REFINED OUTCOME & LEGAL REFERENCES AUDIT

OUTCOME DISTRIBUTION
outcome_refined
إلزام                                                   5731
غير محدد                                                4405
رفض الدعوى أو الطلب                                     1398
عدم قبول الدعوى                                          346
عدم الاختصاص                                             235
عدم جواز نظر الدعوى                                      114
إلزام | فسخ أو إنهاء                                      83
إلزام | رفض الدعوى أو الطلب                               63
فسخ أو إنهاء                                              31
إلزام | عدم قبول الدعوى                                   28
إلزام | عدم الاختصاص                                       7
رفض الدعوى أو الطلب | فسخ أو إنهاء                         6
ثبوت حق أو شراكة                                           5
إلزام | رفض الدعوى أو الطلب | فسخ أو إنهاء                 4
صلح                                                        

In [15]:
# ============================================================
# ESNAD - LEGAL CONTEXT & OUTCOME PRECISION V2
# لا يعيد بناء embeddings أو FAISS
# ============================================================

import pandas as pd
import re
import json

# ------------------------------------------------------------
# 1) Prepare source
# ------------------------------------------------------------

source_df = df_clean.copy().reset_index(drop=True)

def normalize_ar(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.translate(str.maketrans(
        "٠١٢٣٤٥٦٧٨٩", "0123456789"
    ))
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ى", "ي").replace("ة", "ه")
    text = text.replace("ؤ", "و").replace("ئ", "ي")
    text = re.sub(r"[ًٌٍَُِّْـ]", "", text)
    return text

# ------------------------------------------------------------
# 2) Correct outcome: distinguish requested rescission
#    from rescission actually ordered by the court
# ------------------------------------------------------------

def refine_rescission_label(outcome_json, evidence_json):
    labels = json.loads(outcome_json)
    evidence = json.loads(evidence_json)

    if "فسخ أو إنهاء" not in labels:
        return labels, evidence

    text = evidence.get("فسخ أو إنهاء", "")
    text = normalize_ar(text)

    # Split operative ruling into clauses
    clauses = re.split(
        r"\s+(?=(?:اولا|ثانيا|ثالثا|رابعا|خامسا)\s*[:：/\-])"
        r"|[؛;.!؟]",
        text
    )

    positive_rescission = False

    for clause in clauses:
        clause = clause.strip()

        if not re.search(
            r"(?:فسخ العقد|فسخ الاتفاق|انهاء العقد|انهاء الاتفاق)",
            clause
        ):
            continue

        negative_context = re.search(
            r"(?:رفض|عدم قبول|عدم اجابة|رد)\s*"
            r"(?:الدعوي|الدعوى|طلب|طلبات)",
            clause
        )

        if not negative_context:
            positive_rescission = True
            break

    if not positive_rescission:
        labels = [x for x in labels if x != "فسخ أو إنهاء"]
        evidence.pop("فسخ أو إنهاء", None)

    return labels, evidence

# Reuse the previous anchored outcome extraction
assert "classify_refined_outcome" in globals(), (
    "شغّلي خلية التدقيق السابقة أولًا."
)

outcome_rows = []

for _, row in source_df.iterrows():
    status, labels, evidence, anchor_count = (
        classify_refined_outcome(row.get("RULING", ""))
    )

    labels, evidence = refine_rescission_label(
        json.dumps(labels, ensure_ascii=False),
        json.dumps(evidence, ensure_ascii=False)
    )

    final_status = " | ".join(labels) if labels else "غير محدد"

    outcome_rows.append({
        "outcome_v2": final_status,
        "outcomes_v2_list": json.dumps(labels, ensure_ascii=False),
        "outcome_v2_evidence": json.dumps(evidence, ensure_ascii=False),
        "ruling_anchor_count": anchor_count
    })

outcomes_v2_df = pd.DataFrame(outcome_rows)
source_df = pd.concat([source_df, outcomes_v2_df], axis=1)

# ------------------------------------------------------------
# 3) Legal systems: distinguish old and current commercial law
# ------------------------------------------------------------

SYSTEM_ALIASES = {
    "نظام المحاكم التجارية": [
        "نظام المحاكم التجارية"
    ],

    "نظام المحكمة التجارية القديم": [
        "نظام المحكمة التجارية",
        "نظام المحكمه التجاريه"
    ],

    "اللائحة التنفيذية لنظام المحاكم التجارية": [
        "اللائحة التنفيذية لنظام المحاكم التجارية",
        "لايحة نظام المحاكم التجارية",
        "اللائحة التنفيذية للمحاكم التجارية"
    ],

    "نظام المرافعات الشرعية": [
        "نظام المرافعات الشرعية"
    ],

    "نظام الشركات": [
        "نظام الشركات"
    ],

    "نظام الإثبات": [
        "نظام الاثبات"
    ],

    "نظام الإفلاس": [
        "نظام الافلاس"
    ],

    "نظام التحكيم": [
        "نظام التحكيم"
    ],

    "نظام التنفيذ": [
        "نظام التنفيذ"
    ],

    "نظام المعاملات المدنية": [
        "نظام المعاملات المدنية"
    ],

    "نظام الأوراق التجارية": [
        "نظام الاوراق التجارية"
    ],

    "نظام العلامات التجارية": [
        "نظام العلامات التجارية"
    ],

    "نظام الأسماء التجارية": [
        "نظام الاسماء التجارية"
    ],

    "نظام حماية حقوق المؤلف": [
        "نظام حماية حقوق المؤلف"
    ],

    "نظام براءات الاختراع": [
        "نظام براءات الاختراع والتصميمات التخطيطية للدارات المتكاملة والاصناف النباتية والنماذج الصناعية",
        "نظام براءات الاختراع"
    ],

    "نظام الوكالات التجارية": [
        "نظام الوكالات التجارية"
    ],

    "نظام الامتياز التجاري": [
        "نظام الامتياز التجاري"
    ],

    "نظام العمل": [
        "نظام العمل"
    ],

    "نظام السجل التجاري": [
        "نظام السجل التجاري"
    ],

    "نظام الاستثمار": [
        "نظام الاستثمار"
    ],

    "نظام التعاملات الإلكترونية": [
        "نظام التعاملات الالكترونية"
    ],

    "نظام التجارة الإلكترونية": [
        "نظام التجارة الالكترونية"
    ],

    "نظام مكافحة التستر": [
        "نظام مكافحة التستر"
    ],

    "نظام مكافحة الغش التجاري": [
        "نظام مكافحة الغش التجاري"
    ],

    "نظام المنافسة": [
        "نظام المنافسة"
    ],

    "نظام ضريبة القيمة المضافة": [
        "نظام ضريبة القيمة المضافة"
    ],

    "نظام الملكية الفكرية": [
        "نظام الملكية الفكرية"
    ]
}

NORMALIZED_ALIASES = {
    system: [normalize_ar(alias) for alias in aliases]
    for system, aliases in SYSTEM_ALIASES.items()
}

# ------------------------------------------------------------
# 4) Split text into local legal contexts
# ------------------------------------------------------------

def split_legal_segments(text):
    text = normalize_ar(text)

    # Keep segments reasonably local.
    segments = re.split(
        r"[\n\r⏎؛;.!؟]+",
        text
    )

    # Very long segments may contain multiple legal references.
    final_segments = []

    for segment in segments:
        segment = segment.strip()
        if not segment:
            continue

        if len(segment) <= 900:
            final_segments.append(segment)
        else:
            for start in range(0, len(segment), 650):
                part = segment[start:start+800].strip()
                if part:
                    final_segments.append(part)

    return final_segments

ARTICLE_PATTERN_V2 = re.compile(
    r"الماده\s*[:：]?\s*"
    r"(?:رقم\s*)?"
    r"[\(（]?\s*([0-9]{1,4})\s*[\)）]?"
    r"(?:\s*/\s*([0-9]{1,3}))?"
)

def systems_in_segment(segment):
    found = []

    for system, aliases in NORMALIZED_ALIASES.items():
        for alias in aliases:
            if alias and alias in segment:
                found.append(system)
                break

    # Avoid treating the old singular law as the current plural law.
    if "نظام المحكمة التجارية القديم" in found:
        found = [
            x for x in found
            if x != "نظام المحاكم التجارية"
        ]

    return list(dict.fromkeys(found))

# ------------------------------------------------------------
# 5) Extract references with same-segment association
# ------------------------------------------------------------

system_records_v2 = []
article_records_v2 = []

for _, row in source_df.iterrows():
    judgment_id = str(row.get("judgment_number", ""))

    for col, source_name in [
        ("REASONING", "أسباب الحكم"),
        ("RULING", "المنطوق/حقل RULING"),
        ("EVENTS", "وقائع القضية")
    ]:
        raw_text = row.get(col, "")
        if pd.isna(raw_text):
            continue

        segments = split_legal_segments(raw_text)

        for segment in segments:
            mentioned_systems = systems_in_segment(segment)

            for system in mentioned_systems:
                system_records_v2.append({
                    "judgment_number": judgment_id,
                    "system": system,
                    "source": source_name,
                    "context": segment[:900]
                })

            for match in ARTICLE_PATTERN_V2.finditer(segment):
                number = match.group(1)
                sub_number = match.group(2)

                article = f"المادة {number}"
                if sub_number:
                    article += f"/{sub_number}"

                # Associate only if the system is mentioned
                # in the same local segment.
                linked_system = (
                    mentioned_systems[0]
                    if len(mentioned_systems) == 1
                    else None
                )

                article_records_v2.append({
                    "judgment_number": judgment_id,
                    "article": article,
                    "system": linked_system,
                    "source": source_name,
                    "systems_in_same_segment": json.dumps(
                        mentioned_systems, ensure_ascii=False
                    ),
                    "context": segment[:900]
                })

systems_v2_df = pd.DataFrame(system_records_v2)
articles_v2_df = pd.DataFrame(article_records_v2)

# ------------------------------------------------------------
# 6) Summaries
# ------------------------------------------------------------

print("=" * 65)
print("ESNAD - PRECISION AUDIT V2")
print("=" * 65)

print("\nOUTCOME DISTRIBUTION")
print(source_df["outcome_v2"].value_counts().to_string())

identified_v2 = (
    source_df["outcome_v2"] != "غير محدد"
).sum()

multiple_v2 = source_df["outcomes_v2_list"].apply(
    lambda x: len(json.loads(x)) > 1
).sum()

print("\nOUTCOME COVERAGE")
print("Identified:", identified_v2)
print("Unidentified:", len(source_df) - identified_v2)
print("Coverage:", round(identified_v2 / len(source_df) * 100, 2), "%")
print("Multiple outcomes:", multiple_v2)

print("\nLEGAL SYSTEMS")
if not systems_v2_df.empty:
    systems_summary_v2 = (
        systems_v2_df.groupby("system")
        .agg(
            mentions=("system", "size"),
            judgments=("judgment_number", "nunique")
        )
        .sort_values("judgments", ascending=False)
    )
    print(systems_summary_v2.to_string())
else:
    print("No systems found.")

print("\nARTICLE REFERENCES BY SYSTEM")
if not articles_v2_df.empty:
    articles_summary_v2 = (
        articles_v2_df.groupby(
            ["article", "system"], dropna=False
        )
        .agg(
            mentions=("article", "size"),
            judgments=("judgment_number", "nunique")
        )
        .reset_index()
        .sort_values(
            ["judgments", "mentions"],
            ascending=False
        )
    )
    print(articles_summary_v2.head(40).to_string(index=False))
else:
    print("No articles found.")

print("\nLINKING QUALITY")
if not articles_v2_df.empty:
    linked_count = articles_v2_df["system"].notna().sum()
    total_count = len(articles_v2_df)

    print("Article mentions:", total_count)
    print("Linked in same segment:", linked_count)
    print("Unlinked or ambiguous:", total_count - linked_count)
    print(
        "Same-segment linking rate:",
        round(linked_count / total_count * 100, 2),
        "%"
    )

# ------------------------------------------------------------
# 7) Important unmatched article samples
# ------------------------------------------------------------

print("\nUNLINKED ARTICLE SAMPLES")
if not articles_v2_df.empty:
    unmatched = articles_v2_df[
        articles_v2_df["system"].isna()
    ]

    if len(unmatched):
        sample = unmatched.sample(
            min(15, len(unmatched)),
            random_state=21
        )

        for _, item in sample.iterrows():
            print("\nJudgment:", item["judgment_number"])
            print("Article:", item["article"])
            print("Source:", item["source"])
            print(
                "Systems in segment:",
                item["systems_in_same_segment"]
            )
            print("Context:", item["context"][:650])

# ------------------------------------------------------------
# 8) Check known examples
# ------------------------------------------------------------

print("\nKNOWN CASE CHECK: 9549/3/ق")
known = source_df[
    source_df["judgment_number"].astype(str).str.contains(
        "9549", na=False
    )
]

if not known.empty:
    for _, row in known.head(3).iterrows():
        print("Outcome V2:", row["outcome_v2"])
        print("Evidence:", row["outcome_v2_evidence"][:1000])
else:
    print("Judgment not found by this number.")

print("\nOLD COMMERCIAL LAW CHECK")
if not articles_v2_df.empty:
    old_law = articles_v2_df[
        articles_v2_df["article"].eq("المادة 443")
    ]
    print("Article 443 mentions:", len(old_law))
    print(
        old_law["system"].value_counts(dropna=False).to_string()
    )

# ------------------------------------------------------------
# 9) Save
# ------------------------------------------------------------

source_df.to_csv(
    "/content/esnad_arabiccr_precision_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

systems_v2_df.to_csv(
    "/content/esnad_legal_systems_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

articles_v2_df.to_csv(
    "/content/esnad_legal_articles_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nFILES SAVED")
print("/content/esnad_arabiccr_precision_v2.csv")
print("/content/esnad_legal_systems_v2.csv")
print("/content/esnad_legal_articles_v2.csv")
print("=" * 65)

ESNAD - PRECISION AUDIT V2

OUTCOME DISTRIBUTION
outcome_v2
إلزام                                         5732
غير محدد                                      4405
رفض الدعوى أو الطلب                           1400
عدم قبول الدعوى                                347
عدم الاختصاص                                   235
عدم جواز نظر الدعوى                            114
إلزام | فسخ أو إنهاء                            82
إلزام | رفض الدعوى أو الطلب                     63
فسخ أو إنهاء                                    31
إلزام | عدم قبول الدعوى                         28
إلزام | عدم الاختصاص                             7
ثبوت حق أو شراكة                                 5
إلزام | رفض الدعوى أو الطلب | فسخ أو إنهاء       4
رفض الدعوى أو الطلب | فسخ أو إنهاء               4
صلح                                              3
رفض الدعوى أو الطلب | ثبوت حق أو شراكة           3
رفض الدعوى أو الطلب | عدم الاختصاص               3
رفض الدعوى أو الطلب | عدم قبول الدعوى            3
إلزام | عدم جواز نظر ا

In [18]:
# ============================================================
# ESNAD - PRECISION AUDIT V3
# Compare ruling sources + strict legal-reference linking
# Requires: df_clean
# Does NOT rebuild embeddings or FAISS
# ============================================================

import pandas as pd
import re
import json
from collections import Counter

assert "df_clean" in globals(), "df_clean غير موجودة. شغّلي تجهيز البيانات أولًا."

df_v3 = df_clean.copy().reset_index(drop=True)

# ------------------------------------------------------------
# 1) Normalization
# ------------------------------------------------------------

def norm(text):
    if text is None:
        return ""

    text = str(text)

    translation_table = str.maketrans({
        "أ": "ا",
        "إ": "ا",
        "آ": "ا",
        "ى": "ي",
        "ة": "ه",
        "ؤ": "و",
        "ئ": "ي",
        "٠": "0",
        "١": "1",
        "٢": "2",
        "٣": "3",
        "٤": "4",
        "٥": "5",
        "٦": "6",
        "٧": "7",
        "٨": "8",
        "٩": "9"
    })

    text = text.translate(translation_table)

    text = re.sub(r"\s+", " ", text)
    return text.strip()

# ------------------------------------------------------------
# 2) Outcome patterns
# ------------------------------------------------------------

OUTCOME_PATTERNS = {
    "إلزام": [
        r"\bبالزام\b",
        r"\bالزام المدعي عليه\b",
        r"\bالزام المدعي عليها\b",
        r"\bبان يدفع\b",
        r"\bبان تدفع\b",
        r"\bبان يؤدي\b",
        r"\bبان تؤدي\b"
    ],
    "رفض الدعوى أو الطلب": [
        r"رفض الدعوي",
        r"رفض طلب",
        r"رد الدعوي",
        r"رد طلب"
    ],
    "عدم قبول الدعوى": [
        r"عدم قبول الدعوي",
        r"بعدم قبول الدعوي"
    ],
    "عدم جواز نظر الدعوى": [
        r"عدم جواز نظر الدعوي",
        r"بعدم جواز نظر الدعوي"
    ],
    "عدم الاختصاص": [
        r"عدم الاختصاص",
        r"عدم اختصاص المحكمه",
        r"عدم اختصاص الدايره"
    ],
    "فسخ أو إنهاء": [
        r"فسخ العقد",
        r"فسخ الاتفاق",
        r"انهاء العقد",
        r"انهاء الاتفاق"
    ],
    "صلح": [
        r"اثبات الصلح",
        r"اعتماد الصلح",
        r"انتهاء الدعوي صلحا",
        r"انتهاء الدعوي صلح"
    ],
    "شطب الدعوى": [
        r"شطب الدعوي",
        r"شطب القضيه"
    ],
    "حل وتصفية": [
        r"حل الشركه",
        r"تصفية الشركه",
        r"حل وتصفية"
    ],
    "ثبوت حق أو شراكة": [
        r"ثبوت الشراكه",
        r"ثبوت حق",
        r"اثبات الشراكه"
    ]
}

OUTCOME_REGEX = {
    label: [re.compile(p) for p in patterns]
    for label, patterns in OUTCOME_PATTERNS.items()
}

RULING_ANCHORS = [
    "حكمت الدايره",
    "حكمت المحكمه",
    "قضت الدايره",
    "قضت المحكمه",
    "لذلك حكمت",
    "فلهذه الاسباب حكمت",
    "عليه حكمت المحكمه",
    "عليه حكمت الدايره",
    "حكمت الدائره"
]

HISTORICAL_CONTEXT = [
    "سبق ان صدر",
    "صدر حكم سابق",
    "الحكم السابق",
    "الحكم الصادر",
    "حصل علي حكم",
    "بموجب الحكم",
    "بموجب صك الحكم",
    "قضي سابقا",
    "قضت المحكمه سابقا"
]

def extract_ruling_windows(text, max_chars=2200):
    text = norm(text)
    windows = []

    for anchor in RULING_ANCHORS:
        start = 0
        while True:
            pos = text.find(anchor, start)
            if pos < 0:
                break

            before = text[max(0, pos-220):pos]

            if not any(marker in before for marker in HISTORICAL_CONTEXT):
                windows.append({
                    "start": pos,
                    "text": text[pos:pos+max_chars]
                })

            start = pos + len(anchor)

    # Remove nearly identical windows
    unique = []
    seen = set()

    for item in windows:
        signature = item["text"][:350]
        if signature not in seen:
            seen.add(signature)
            unique.append(item)

    return unique

def classify_windows(windows):
    evidence = {}

    for label, patterns in OUTCOME_REGEX.items():
        matches = []

        for window in windows:
            content = window["text"]

            if any(pattern.search(content) for pattern in patterns):
                matches.append(content[:800])

        if matches:
            evidence[label] = matches[:3]

    labels = list(evidence.keys())
    return labels, evidence

def analyze_outcome_source(text):
    windows = extract_ruling_windows(text)
    labels, evidence = classify_windows(windows)

    return {
        "labels": labels,
        "evidence": evidence,
        "anchor_count": len(windows)
    }

# Analyze RULING and full case_text separately
ruling_analysis = []
case_analysis = []

for _, row in df_v3.iterrows():
    ruling_analysis.append(
        analyze_outcome_source(row.get("RULING", ""))
    )
    case_analysis.append(
        analyze_outcome_source(row.get("case_text", ""))
    )

df_v3["outcomes_from_ruling"] = [
    json.dumps(x["labels"], ensure_ascii=False)
    for x in ruling_analysis
]

df_v3["outcomes_from_case_text"] = [
    json.dumps(x["labels"], ensure_ascii=False)
    for x in case_analysis
]

df_v3["ruling_anchor_count_v3"] = [
    x["anchor_count"] for x in ruling_analysis
]

df_v3["case_anchor_count_v3"] = [
    x["anchor_count"] for x in case_analysis
]

def merge_outcomes(a, b):
    # Preserve source-level differences for audit.
    merged = list(dict.fromkeys(a["labels"] + b["labels"]))
    return merged

merged_outcomes = [
    merge_outcomes(a, b)
    for a, b in zip(ruling_analysis, case_analysis)
]

df_v3["outcomes_v3_list"] = [
    json.dumps(x, ensure_ascii=False)
    for x in merged_outcomes
]

df_v3["outcome_v3"] = [
    " | ".join(x) if x else "غير محدد"
    for x in merged_outcomes
]

df_v3["outcome_source_disagreement"] = [
    set(a["labels"]) != set(b["labels"])
    for a, b in zip(ruling_analysis, case_analysis)
]

# ------------------------------------------------------------
# 3) Legal source dictionary
#    Singular old name is kept separate from current plural name.
# ------------------------------------------------------------

LEGAL_ALIASES = {
    "نظام المحاكم التجارية": [
        "نظام المحاكم التجارية"
    ],
    "نظام المحكمة التجارية القديم": [
        "نظام المحكمة التجارية"
    ],
    "اللائحة التنفيذية لنظام المحاكم التجارية": [
        "اللائحة التنفيذية لنظام المحاكم التجارية",
        "لايحة نظام المحاكم التجارية",
        "اللائحة التنفيذية للمحاكم التجارية"
    ],
    "نظام المرافعات الشرعية": [
        "نظام المرافعات الشرعية"
    ],
    "نظام الشركات": [
        "نظام الشركات"
    ],
    "نظام الإثبات": [
        "نظام الاثبات"
    ],
    "نظام الإفلاس": [
        "نظام الافلاس"
    ],
    "نظام التحكيم": [
        "نظام التحكيم"
    ],
    "نظام التنفيذ": [
        "نظام التنفيذ"
    ],
    "نظام المعاملات المدنية": [
        "نظام المعاملات المدنية"
    ],
    "نظام الأوراق التجارية": [
        "نظام الاوراق التجارية"
    ],
    "نظام العلامات التجارية": [
        "نظام العلامات التجارية"
    ],
    "نظام الأسماء التجارية": [
        "نظام الاسماء التجارية"
    ],
    "نظام حماية حقوق المؤلف": [
        "نظام حماية حقوق المؤلف"
    ],
    "نظام براءات الاختراع": [
        "نظام براءات الاختراع والتصميمات التخطيطية للدارات المتكاملة والاصناف النباتية والنماذج الصناعية",
        "نظام براءات الاختراع"
    ],
    "نظام الوكالات التجارية": [
        "نظام الوكالات التجارية"
    ],
    "نظام الامتياز التجاري": [
        "نظام الامتياز التجاري"
    ],
    "نظام العمل": [
        "نظام العمل"
    ],
    "نظام السجل التجاري": [
        "نظام السجل التجاري"
    ],
    "نظام الاستثمار": [
        "نظام الاستثمار"
    ],
    "نظام التعاملات الإلكترونية": [
        "نظام التعاملات الالكترونية"
    ],
    "نظام التجارة الإلكترونية": [
        "نظام التجارة الالكترونية"
    ],
    "نظام مكافحة التستر": [
        "نظام مكافحة التستر"
    ],
    "نظام مكافحة الغش التجاري": [
        "نظام مكافحة الغش التجاري"
    ],
    "نظام المنافسة": [
        "نظام المنافسة"
    ],
    "نظام ضريبة القيمة المضافة": [
        "نظام ضريبة القيمة المضافة"
    ],
    "نظام الملكية الفكرية": [
        "نظام الملكية الفكرية"
    ],
    "مجلة الأحكام العدلية": [
        "مجلة الاحكام العدلية"
    ]
}

# Compile aliases, longest first
ALIAS_LOOKUP = []

for canonical, aliases in LEGAL_ALIASES.items():
    for alias in aliases:
        ALIAS_LOOKUP.append((norm(alias), canonical))

ALIAS_LOOKUP.sort(key=lambda x: len(x[0]), reverse=True)

# ------------------------------------------------------------
# 4) Detect legal source mentions with exact spans
# ------------------------------------------------------------

def find_legal_sources(text):
    text = norm(text)
    found = []

    for alias, canonical in ALIAS_LOOKUP:
        start = 0

        while True:
            pos = text.find(alias, start)
            if pos < 0:
                break

            found.append({
                "canonical": canonical,
                "alias": alias,
                "start": pos,
                "end": pos + len(alias)
            })

            start = pos + len(alias)

    # Remove duplicate/overlapping alias hits
    found.sort(key=lambda x: (x["start"], -(x["end"]-x["start"])))

    clean = []
    for item in found:
        overlaps = any(
            item["start"] < old["end"] and item["end"] > old["start"]
            for old in clean
        )
        if not overlaps:
            clean.append(item)

    return clean

# ------------------------------------------------------------
# 5) Article extraction
#    Supports: المادة 16، المادة (16)، المادة 57/2
# ------------------------------------------------------------

ARTICLE_RE = re.compile(
    r"الماده\s*"
    r"(?:رقم\s*)?"
    r"[\(（]?\s*([0-9]{1,4})\s*[\)）]?"
    r"(?:\s*/\s*([0-9]{1,3}))?"
)

def split_segments(text):
    # Split into sentence-like segments, retaining useful context.
    parts = re.split(r"(?<=[؛;.!؟])\s+|[\n\r]+", text)
    return [p.strip() for p in parts if p.strip()]

def extract_legal_references(text, source_name, judgment_id):
    text = norm(text)
    if not text:
        return [], []

    system_mentions = find_legal_sources(text)
    article_records = []
    system_records = []

    for mention in system_mentions:
        system_records.append({
            "judgment_number": judgment_id,
            "system": mention["canonical"],
            "source": source_name,
            "context": text[max(0, mention["start"]-180):
                            min(len(text), mention["end"]+180)]
        })

    segments = split_segments(text)

    for match in ARTICLE_RE.finditer(text):
        number = match.group(1)
        subsection = match.group(2)

        article = f"المادة {number}"
        if subsection:
            article += f"/{subsection}"

        pos = match.start()

        # Find the segment containing the article.
        segment = next(
            (s for s in segments if s in text[max(0, pos-100):pos+500]),
            ""
        )

        # Use a local context around the article.
        local_start = max(0, pos-240)
        local_end = min(len(text), match.end()+300)
        context = text[local_start:local_end]

        # Only link when the source name is in the same local context.
        nearby = []

        for system in system_mentions:
            distance = min(
                abs(system["start"] - match.start()),
                abs(system["end"] - match.end())
            )

            if distance <= 240:
                nearby.append((distance, system["canonical"]))

        nearby.sort(key=lambda x: x[0])

        linked_system = None
        linking_reason = "لا يوجد اسم نظام قريب بما يكفي"

        if nearby:
            closest_distance = nearby[0][0]
            closest_systems = list(dict.fromkeys(
                system for distance, system in nearby
                if distance == closest_distance
            ))

            if len(closest_systems) == 1:
                linked_system = closest_systems[0]
                linking_reason = "اسم النظام الأقرب في السياق"
            else:
                linking_reason = "أكثر من نظام بالمسافة نفسها"

        # Flag known non-system references separately.
        if "مجلة الاحكام العدلية" in context:
            if linked_system is None:
                linked_system = "مجلة الأحكام العدلية"
                linking_reason = "مرجع صريح لمجلة الأحكام العدلية"

        article_records.append({
            "judgment_number": judgment_id,
            "article": article,
            "system": linked_system,
            "source": source_name,
            "linking_reason": linking_reason,
            "context": context
        })

    return system_records, article_records

# ------------------------------------------------------------
# 6) Extract from separate dataset fields
# ------------------------------------------------------------

all_systems = []
all_articles = []

for _, row in df_v3.iterrows():
    judgment_id = str(row.get("judgment_number", ""))

    for col, source_name in [
        ("REASONING", "أسباب الحكم"),
        ("RULING", "حقل RULING"),
        ("EVENTS", "وقائع القضية"),
        ("case_text", "النص الكامل")
    ]:
        sys_records, article_records = extract_legal_references(
            row.get(col, ""),
            source_name,
            judgment_id
        )

        all_systems.extend(sys_records)
        all_articles.extend(article_records)

systems_v3 = pd.DataFrame(all_systems)
articles_v3 = pd.DataFrame(all_articles)

# ------------------------------------------------------------
# 7) Summaries
# ------------------------------------------------------------

if not systems_v3.empty:
    systems_summary_v3 = (
        systems_v3.groupby("system")
        .agg(
            mentions=("system", "size"),
            judgments=("judgment_number", "nunique")
        )
        .sort_values("judgments", ascending=False)
    )
else:
    systems_summary_v3 = pd.DataFrame()

if not articles_v3.empty:
    articles_summary_v3 = (
        articles_v3.groupby(["article", "system"], dropna=False)
        .agg(
            mentions=("article", "size"),
            judgments=("judgment_number", "nunique")
        )
        .reset_index()
        .sort_values(["judgments", "mentions"], ascending=False)
    )

    linked_count = articles_v3["system"].notna().sum()
    unlinked_count = articles_v3["system"].isna().sum()
else:
    articles_summary_v3 = pd.DataFrame()
    linked_count = 0
    unlinked_count = 0

# ------------------------------------------------------------
# 8) Outcome audit report
# ------------------------------------------------------------

print("=" * 65)
print("ESNAD - PRECISION AUDIT V3")
print("=" * 65)

print("\nOUTCOME DISTRIBUTION")
print(df_v3["outcome_v3"].value_counts().to_string())

identified = (df_v3["outcome_v3"] != "غير محدد").sum()
multiple = sum(len(x) > 1 for x in merged_outcomes)
disagreements = df_v3["outcome_source_disagreement"].sum()

print("\nOUTCOME COVERAGE")
print("Total judgments:", len(df_v3))
print("Identified:", identified)
print("Unidentified:", len(df_v3)-identified)
print("Coverage:", round(identified/len(df_v3)*100, 2), "%")
print("Multiple outcomes:", multiple)
print("RULING vs case_text disagreements:", disagreements)

print("\nSOURCE COMPARISON")
print("RULING identified:",
      sum(bool(x["labels"]) for x in ruling_analysis))
print("case_text identified:",
      sum(bool(x["labels"]) for x in case_analysis))

print("\nDISAGREEMENT SAMPLES")
disagreement_rows = df_v3[
    df_v3["outcome_source_disagreement"]
].head(15)

for _, row in disagreement_rows.iterrows():
    print("\nJudgment:", row.get("judgment_number"))
    print("RULING:", row["outcomes_from_ruling"])
    print("case_text:", row["outcomes_from_case_text"])
    print("Merged:", row["outcomes_v3_list"])

# ------------------------------------------------------------
# 9) Legal reference audit report
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("LEGAL SYSTEMS")
print("=" * 65)

if not systems_summary_v3.empty:
    print(systems_summary_v3.to_string())
else:
    print("No legal systems found.")

print("\n" + "=" * 65)
print("ARTICLE REFERENCES BY SYSTEM")
print("=" * 65)

if not articles_summary_v3.empty:
    print(articles_summary_v3.head(50).to_string(index=False))

print("\nLINKING QUALITY")
print("Article mentions:", len(articles_v3))
print("Linked:", linked_count)
print("Unlinked:", unlinked_count)

if len(articles_v3):
    print("Linking rate:",
          round(linked_count/len(articles_v3)*100, 2), "%")

# ------------------------------------------------------------
# 10) Important known-case checks
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("KNOWN CASE CHECKS")
print("=" * 65)

for case_id in ["9549/3/ق", "285/8/ق", "5128/4/ق"]:
    matches = df_v3[
        df_v3["judgment_number"].astype(str) == case_id
    ]

    if not matches.empty:
        row = matches.iloc[0]
        print("\nJudgment:", case_id)
        print("RULING outcomes:", row["outcomes_from_ruling"])
        print("case_text outcomes:", row["outcomes_from_case_text"])
        print("Merged:", row["outcomes_v3_list"])

if not articles_v3.empty:
    for case_id in ["285/8/ق", "5128/4/ق", "3414"]:
        sample = articles_v3[
            articles_v3["judgment_number"] == case_id
        ]

        if not sample.empty:
            print("\nLegal references for:", case_id)
            print(sample[[
                "article", "system", "source", "linking_reason"
            ]].head(12).to_string(index=False))

# ------------------------------------------------------------
# 11) Export
# ------------------------------------------------------------

df_v3.to_csv(
    "/content/esnad_arabiccr_precision_v3.csv",
    index=False,
    encoding="utf-8-sig"
)

systems_v3.to_csv(
    "/content/esnad_legal_systems_v3.csv",
    index=False,
    encoding="utf-8-sig"
)

articles_v3.to_csv(
    "/content/esnad_legal_articles_v3.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 65)
print("FILES SAVED")
print("/content/esnad_arabiccr_precision_v3.csv")
print("/content/esnad_legal_systems_v3.csv")
print("/content/esnad_legal_articles_v3.csv")
print("=" * 65)

ESNAD - PRECISION AUDIT V3

OUTCOME DISTRIBUTION
outcome_v3
إلزام                                                                                                5598
غير محدد                                                                                             4301
رفض الدعوى أو الطلب                                                                                  1373
عدم قبول الدعوى                                                                                       351
عدم الاختصاص                                                                                          248
عدم جواز نظر الدعوى                                                                                   118
إلزام | رفض الدعوى أو الطلب                                                                            94
إلزام | فسخ أو إنهاء                                                                                   78
فسخ أو إنهاء                                                                                